<a href="https://colab.research.google.com/github/Oaimtac/farm-soccer/blob/main/%E5%BF%83%E9%9B%BB%E5%9C%96%E8%A8%8A%E8%99%9F%E7%96%B2%E5%8B%9E%E5%88%86%E6%9E%90%E5%8E%9F%E7%90%86%E8%88%87%E5%AF%A6%E4%BD%9C(II)_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **心電圖訊號疲勞分析原理與實作(II)_1**

# 0. 先收錄一些會用到的公式和函式吧！

In [ ]:
from scipy.fft import fft, fftfreq, ifft #從scipy.fft函式庫中，引入頻域轉換公式fft, fftfreq, ifft
import plotly.graph_objects as go     #引入plotly.graph_objects函式庫，命名為go
import numpy as np             #引入numpy函式庫，命名為np
import pandas as pd            #引入pandas函式庫，命名為np
from google.colab import files      #從google colab函式庫中，引入files函式

# 1.將上次Trianswer紀錄下來「我的心電訊號」再放進來一次吧！

In [ ]:
#先將資料夾中的「我的心電訊號」檔案放到Google雲端處理器中
uploaded = files.upload()            #用uploaded來處理上傳檔案程序
for fn in uploaded.keys():           #將選取所有上傳檔案的名字印出
  print('你已經上傳了','"{name}" '.format(
      name=fn, length=len(uploaded[fn])))

# 2.看看自己的心電訊號長什麼樣子吧！


In [ ]:
df = pd.read_csv('我的心電訊號.txt') #以pandas的read_csv即可讀取檔案
data = np.array(df)          #將讀取到的檔案換成我們習慣的numpy陣列來處理
data2 = data[:,0]           #指定data中的第一行資料，建立陣列data2

fig = go.Figure()      #建立一個圖形物件
fig.add_trace(go.Scatter(   #新增一條線條在此圖形
    y=data2,        #新增一條線條，將取得的data2畫出
))
fig.update_layout(                  #更新圖形的說明
    title="我的心電訊號的時域波形",        #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig.show()                      #顯示圖形

# 3.利用上一章節的強大濾波器，濾出R波特化的「我的心電訊號」

> 利用前面章節所學到的強大濾波器，將「我的心電訊號」中的R波訊號濾出：R波能量 (15Hz 到 40Hz)

> 往下再利用找波峰函式把R波位置找出來，進一步找出R波間隔



In [ ]:
#強大的濾波器，不需要再轉換到頻域即可快速濾除雜訊，還能達到一樣的效果！
from scipy.signal import butter, filtfilt #從scipy.signal函式庫中，引入濾波器函式butter, filtfilt
#訊號處理常用的濾波器，可直接根據定義的頻域能量範圍，將正確的訊號過濾出來
#使用方法需要提供(時域訊號、資料取樣頻率、特定保留頻率起點、特定保留頻率終點)
def super_filter(data, frequency, save_frequency_start, save_frequency_end):   #super_filter(時域訊號, 資料取樣頻率, 特定保留頻率起點, 特定保留頻率終點)
  b, a = butter(3, [save_frequency_start, save_frequency_end], fs=frequency, btype='band')
  y = filtfilt(b, a, data)
  return y

In [ ]:
data3 = data[:,0]           #指定data中的第一行資料，重新建立一筆陣列data3
period = 1/500     #宣告資料取樣週期參數
frequency = 500     #宣告資料取樣頻率參數
data4 = super_filter(data3, period, 15, 35) #將時域訊號, 資料取樣頻率, 心電訊號R波保留頻率起點, 心電訊號R波保留頻率終點代入強大的濾波器(super filter)

fig2 = go.Figure()                #建立一個圖形物件
fig2.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=data4,                #新增一條線條，，將轉換訊號的時域部分畫出
))
fig2.update_layout(                  #更新圖形的說明
    title="「我的心電訊號」經濾波器處理濾出R段後的R波特化波形", #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig2.show()                 #顯示圖形



> 借助強大的find_peaks找波峰函式來為我們找到R波波峰位置

> find_peaks的參數：(資料, 波峰認定資料最低限值, 波峰之間間隔資料點最低限值)


In [ ]:
from scipy.signal import find_peaks #從scipy.signal函式庫中，引入找波峰函式find_peaks

#Height參數設定：請先觀察R波特化的資料秀出的波形，Height應該設定為多少才合適？

#Distance參數設定：
#1.若資料錄製時設定Middle，資料記錄頻率為500Hz，資料記錄週期為0.002秒
#2.假設人體每次心跳最快每分鐘也不可能超過210(次/分鐘)的話，換算為秒數單位的話就是3.5(次/秒)
#3.每個R波之間間距最低秒數，就可以直接取3.5的倒數，近似為0.28秒
#4.已知資料記錄週期為0.002秒，即可得知資料點最低限值為0.28/0.002=140

height=5    #宣告波峰認定資料最低限值
distance=140  #宣告,波峰之間間隔資料點最低限值
peak_list_x = find_peaks(data4, height=height, distance=distance)[0]  #指定其x軸的位置為波峰偵測位置
peak_list_y = [data4[j] for j in peak_list_x]              #指定其y軸的位置為波峰偵測位置時的data5對應數值(R波特化)

print("印出波峰偵測位置的第一個點：",peak_list_x[0])
print("印出波峰偵測位置第一點的data4對應數值：",peak_list_y[0])

In [ ]:
#畫出濾出R段後的波形與波峰偵測位置
fig2 = go.Figure()                #建立一個圖形物件
fig2.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=data4,                #新增一條線條，，將轉換訊號的時域部分畫出
    name='我的心電訊號(濾出R波段)'    #幫此線條命名
))
fig2.add_trace(go.Scatter(           #新增一條線條在此圖形
    x=peak_list_x,              #指定其x軸的位置
    y=peak_list_y,              #指定其y軸的位置
    mode='markers',             #定義此線段不連線，僅畫出有標記的位置
    marker=dict(
        color='red',           #幫此標記以紅色標記
    ),
    name='分析之波峰位置'          #幫此線條命名
))
fig2.update_layout(                  #更新圖形的說明
    title="我的心電訊號(濾出R波段)與偵測波峰", #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)
fig2.show()                 #顯示圖形

In [ ]:
#將每個數值印出，確認波峰陣列資料確實就是波峰偵測到的數值
print('每個心跳波峰是在資料位置',peak_list_x)

In [ ]:
#利用Numpy函式庫定義非常方便的函式(diff: differnece差距)
#將peak_list_x陣列直接代入，獲得和R波峰間距

peak_peak_list = np.diff(peak_list_x) #兩兩波峰相減計算波峰之間的距離

print("直接引用numpy函式庫的diff函式，得到的波峰位置間距：", peak_peak_list)
print("此方法得出的波峰位置間距的資料數為：", len(peak_peak_list))

#4. 4赫茲再取樣法



> 要將所有波峰重新排列，可以執行下面步驟：

0.   核心概念：4赫茲再取樣就是將原本每秒500個資料點，改為4個資料點，並用波峰間距數值取代
1.   首先，將心電訊號的最後一個波峰位置，減去第一個波峰位置，就能得知心電訊號在首尾波峰之間總共有幾個資料點，宣告為peak_list_length。
2.   將peak_list_length除以500，就能得知這段期間心電訊號的總秒數
3.   再將這個總秒數乘以4，得知在4Hz頻率下，新建立的波峰間距陣列的資料總長需要設定多少
4.   最後，設定一個迴圈，將心電訊號波峰的資料以125(500除以4)的速度往前推移，推移的同時，將波峰間距資料依序輸入進4赫茲的波峰間距陣列。


In [ ]:
#有意願挑戰的同學可以自行接著嘗試根據說明，完成這段20行以內可完成的4赫茲再取樣陣列






In [ ]:
#導師講解1
peak_list_x_last = peak_list_x[-1]           #一個陣列中的最後一個數值，可以用array[-1]取得
peak_list_length = peak_list_x[-1] - peak_list_x[0] #將心電訊號的最後一個波峰位置，減去第一個波峰位置，就能得知心電訊號在首尾波峰之間總共有幾個資料點，宣告為peak_list_length。
print("心電訊號的波峰位置陣列為：：",peak_list_x)
print("心電訊號在首尾波峰之間的資料總點數：", peak_list_length)

In [ ]:
#導師講解2,3
data4_seconds = peak_list_length/500              #將心電訊號首尾波峰之間資料總數除以500，得出訊號總秒數
new_4Hz_peaks_peaks_list_x_length =  int(data4_seconds*4)  #將總秒數乘以4，得知新建立的波峰間距陣列的資料總長需要設定多少

print("首尾波峰之間心電訊號的總秒數：", data4_seconds)
print("新的波峰間距陣列資料總長度設定為：", new_4Hz_peaks_peaks_list_x_length)

In [ ]:
#導師講解4-1
n = 0           #n設定為0，對陣列來說代表從第一個資料點開始
pointer = peak_list_x[n] #從第一個波峰位置開始推移
pointer_speed = 125    #從500Hz的原始資料紀錄頻率，再取樣為4Hz頻率的資料，原始資料每推移125即可執行一次4Hz的資料紀錄，紀錄的數值為波峰間距
pointer_counter = n+1   #推移的過程，需要與下一個波峰位置比較，若發現已經超過下一個波峰位置，就要將紀錄內容更換到正確的波峰間距數值

new_4Hz_peaks_peaks_list = np.zeros(new_4Hz_peaks_peaks_list_x_length) #建立一個空的4Hz波峰間距陣列
counter = 0                                 #定位空陣列的資料位置，幫一個空陣列資料填入數值後要記得加一

In [ ]:
#導師講解4-2
for i in range(new_4Hz_peaks_peaks_list_x_length):    #用迴圈把正確的波峰間距，填入預設空的4Hz波峰間距陣列
  if(pointer > peak_list_x[pointer_counter]):  #每次都要先與下一個波峰位置比較，若發現已經超過下一個波峰位置，就要將紀錄內容更換到對應的波峰間距數值
    pointer_counter = pointer_counter + 1  #若有發現超過，就要加一，來更換到正確的波峰間距數值
  pointer = pointer + pointer_speed       #以預設好的推移速度pointer_speed，將pointer向前推移

  new_4Hz_peaks_peaks_list[counter] = peak_peak_list[pointer_counter - 1]   #將正確的波峰間距填入空的4Hz波峰間距陣列；pointer_counter從1開始，所以這邊要減一，從波峰間距的起始位置(0)開始填入
  counter = counter + 1                              #定位空陣列的資料位置，幫一個空陣列資料填入數值後要記得加一

print("4Hz再取樣完成後的波峰間距陣列為：")
print(new_4Hz_peaks_peaks_list)

In [ ]:
#畫出4Hz再取樣後的波峰間距陣列波形
fig = go.Figure()                #建立一個圖形物件
fig.add_trace(go.Scatter(             #新增一條線條在此圖形
    y=new_4Hz_peaks_peaks_list,        #新增一條線條，將轉換訊號的畫出
    mode='markers',               #定義此線段不連線，僅畫出有標記的位置
))
fig.update_layout(                  #更新圖形的說明
    title="我的心電訊號(4Hz再取樣的波峰陣列)", #幫這張圖形物件命名
    font=dict(                  #設定圖形名稱的文字
        family="Courier New, monospace",  #字體類型設定：Courier New, monospace
        size=20,               #字體大小設定：20
        color="RebeccaPurple"         #字體顏色設定：RebeccaPurple
    )
)

print("前面預先算出的波峰位置間距：", peak_peak_list)
fig.show()                 #顯示圖形